In [ ]:
### 추천 시스템에서 사용하는 평가 지표

## 1. RMSE  : np.sqrt( mean_squared_error( trues, preds ) )       or    mean_squared_error( trues, preds ) ** 0.5
    - trues, preds 는 np.array() 형태여야 한다. tolist() 함수 사용

## 2. MAE : mean_absolute_error( trues, preds )

## 3. Recall@K      : (Relevant item in top-K items) / Total number of relevant items
## 4. Precission@K  : (Relevant item in top-K items) / K

## 5. DCG & NDCG
    - DCG : sum of { ( relevanc of item(i) ) / log2(i + 1) } for i=1 to K
    - IDCG : sum of { ( relevanc of item(i) in ideal list ) / log2(i + 1) } for i=1 to K
    - NDCG = DCG / IDCG

In [ ]:
### NCF (Neural Collaborative Filtering)
    - MovieLens 데이터 사용 (user, movie, rating)
    - user 와 movie 의 id 를 순차적에게 LabelEncoder() 로 변경
    - user 와 movie 의 관계를 rating 을 이용해서 Linear layer 로 학습 (FC, ReLU, FC)
    - 원래 비어있던 user 와 movie 관계를 모델로 예측 가능함
    - Loss : RMSE, Recall@K, Precision@K

In [ ]:
### 1. Data Preprocessing

class MovieLens:
  def __init__(self, users, movies, ratings):
    self.users = users
    self.movies = movies
    self.ratings = ratings

  def __len__(self):
    return len(self.users)

  def __getitem__(self,item):
    users = self.users[item]
    movies = self.movies[item]
    ratings = self.ratings[item]

    return {'users': torch.tensor(users, dtype = torch.long).to(device),
            'movies': torch.tensor(movies, dtype = torch.long).to(device),
            'ratings': torch.tensor(ratings, dtype=torch.long).to(device)}

# convert actual index for user and items as consecutive integer
lbl_user = preprocessing.LabelEncoder()
lbl_movie = preprocessing.LabelEncoder()

df.userId = lbl_user.fit_transform(df.userId.values)
df.movieId = lbl_movie.fit_transform(df.movieId.values)

# devide original dataframe into train, test dataframe (9:1)
df_train, df_test = model_selection.train_test_split(df, test_size=0.1, random_state=42, stratify=df.rating.values)

train_dataset = MovieLens(users = df_train.userId.values, movies = df_train.movieId.values, ratings = df_train.rating.values)
test_dataset = MovieLens(users = df_test.userId.values, movies = df_test.movieId.values, ratings = df_test.rating.values)

In [ ]:
### 2. NCF 모델 class

### Embedding layer + NCF layer
# 2-layer(including output layer)

# hidden layer + active layer(RELU) + output
# latent vector dim: 32
# input dim: 64

class Neural_Collaborative_Filtering(nn.Module):
  def __init__(self, n_users, n_movies):
    '''
    n_users = # of users
    n_movies = # of movies
    '''
    super().__init__()

    # convert all users and items into 32-dim learnable latent factors(vectors)
    self.user_embedding = nn.Embedding(n_users, 32)
    self.movie_embedding = nn.Embedding(n_movies, 32)

    # Neural network for rating prediction
    self.fc1 = nn.Linear(64,32) # 64 = user_i embedding + movie_j embedding
    self.relu = nn.ReLU()
    self.fc2= nn.Linear(32,1)

  def forward(self, users, movies, ratings = None):
    user_embedding = self.user_embedding(users)
    movie_embedding = self.movie_embedding(movies)
    input_embedding = torch.cat( [ user_embedding, movie_embedding ], dim = 1 )

    hidden_feature = self.fc1(input_embedding)
    hidden_feature = self.relu(hidden_feature)

    output = self.fc2(hidden_feature) # output is prediction

    return output

In [ ]:
### 3. Training NFC 모델

# set batch
train_loader = DataLoader(dataset = train_dataset, batch_size=128, shuffle=True, drop_last=False)
test_loader = DataLoader(dataset = test_dataset, batch_size=128, shuffle=True, drop_last = False)

# model instance generation
model = Neural_Collaborative_Filtering(n_users = len(lbl_user.classes_), n_movies = len(lbl_movie.classes_)).to(device)

# Optimizer, Objective function
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
loss_func = nn.MSELoss( reduction= 'mean' )

epochs = 5
total_loss = 0
iter_cnt = 0
all_losses_list = []

model.train()

for epoch in range(1, epochs+1):
  total_loss = 0
  epoch_check = 0

  for i, train_data in enumerate(train_loader):
    '''
    train_data = {'users':[], 'items':[], 'ratings':[]}
    '''
    optimizer.zero_grad()

    batch_size = len(train_data['users'])
    prediction = model(train_data['users'], train_data['movies'])
    ground_truth = train_data['ratings'].view(batch_size,-1).to(torch.float32)

    loss = loss_func(prediction, ground_truth)
    total_loss = total_loss + (loss.item() * batch_size)
    
    loss.backward()
    optimizer.step()

    iter_cnt = iter_cnt + 1
    epoch_check += batch_size

    if iter_cnt % 100 == 0 and iter_cnt != 0:
      avg_iter_loss = loss.item()
      batch_num = int((iter_cnt/100) % 7) if int((iter_cnt/100) % 7) else 7
      print(f"epoch {epoch} - (batch {batch_num}) loss : {(avg_iter_loss)}")

    if epoch_check % (batch_size * len(train_loader)) == 0 and epoch_check != 0:
      avg_loss = total_loss / epoch_check
      print(f"Epoch {epoch} Avg_loss : {avg_loss}")
      all_losses_list.append(avg_loss)

In [ ]:
### 4. Evaluation

## 4.1 RMSE

from sklearn.metrics import mean_squared_error

model_output_list = []
target_rating_list = []

model.eval()

with torch.no_grad():
  for i, batched_data in enumerate(test_loader):
    model_output = model( batched_data['users'], batched_data['movies'] )
    model_output_batch = model_output.cpu().numpy().squeeze(axis=1).tolist()
    model_output_list += (model_output_batch)

    target_rating = batched_data['ratings']
    target_rating_batch = target_rating.cpu().numpy().tolist()
    target_rating_list += target_rating_batch

mse = mean_squared_error( target_rating_list, model_output_list )
rms = np.sqrt(mse)
print(f"rms: {rms}")


## 4.2 Recall@K, Prediction@K

from collections import defaultdict

user_est_true = defaultdict(list)

with torch.no_grad():
  for i, batched_data in enumerate(test_loader):
    users = batched_data['users']
    movies = batched_data['movies']
    ratings = batched_data['ratings']

    model_output = model(batched_data['users'], batched_data["movies"])

    for i in range(len(users)):
      user_id = users[i].item()
      movie_id = movies[i].item()
      pred_rating = model_output[i][0].item()
      true_rating = ratings[i].item()

      user_est_true[user_id].append( (pred_rating, true_rating) )

with torch.no_grad():
  precisions = dict()
  recalls = dict()

  # recall@K
  k= 10
  threshold = 3.5 # relevant item criterion

  for user_id, user_ratings in user_est_true.items():
    user_ratings.sort(key=lambda x: x[0], reverse =True)

    # get the number for real relevant items = denominator of recall@k
    n_real_relevant = sum((true_r >= threshold) for (_, true_r) in user_ratings)

    # k recommended ratings
    recommended_k = user_ratings[:k]

    # get the number of recommented item that is actually relevant with real relevant.
    n_real_relevant_in_top_k = sum((true_r >= threshold) for (est, true_r) in recommended_k)

    # precision@k
    precisions[user_id] = n_real_relevant_in_top_k / len(recommended_k)

    # recall@k
    if n_real_relevant:
      recalls[user_id] = n_real_relevant_in_top_k / n_real_relevant
    #else:
    #  recalls[user_id] = 0     # 최종 평균 낸 Recall 계산 시에 0 값이 들어가서 평균을 망가지게 하므로 넣지 안는게 더 좋음

# Precision and recall can then be averaged over all users
print(f"precision @ {k}: {sum(prec for prec in precisions.values()) / len(precisions)}")
print(f"recall @ {k} : {sum(rec for rec in recalls.values()) / len(recalls)}")     

In [ ]:
# ============================================================================================================================

In [ ]:
### NGCF (Neural Graph Collaborative Filtering)
    - Graph 를 이용한 user - item 간의 상호작용(message) 를 layer 를 거쳐가면서 학습에 사용함
    - Loss : BPR (Bayesian Personalized Ranking) Loss 사용

In [ ]:
### 1. Data Preprocessing

# Transform UserID and MovieID into sequential indices
user_encoder = {user: idx for idx, user in enumerate(rating_df['userId'].unique())}
movie_encoder = {movie: idx for idx, movie in enumerate(rating_df['movieId'].unique())}

rating_df['userId'] = rating_df['userId'].map(user_encoder)
rating_df['movieId'] = rating_df['movieId'].map(movie_encoder)

"""
    아래 코드와 동일한 기능
    lbl_user = preprocessing.LabelEncoder()
    lbl_movie = preprocessing.LabelEncoder()

    rating_df.userId = lbl_user.fit_transform( rating_df.userId.values )
    rating_df.movieId = lbl_movie.fit_transform( rating_df.movieId.values )
"""

num_users = len(user_encoder)
num_movies = len(movie_encoder)

In [ ]:
### 2. Create Graph & split data

# Create Edge for Graph
# We generate edge between user and movie when user rates movie higher than (or equal to) 1
# Adjacency matrix 대신 edge_index를 넣어서도 진행가능하다.

def create_edge_index(df, rating_threshold=1.0):
    src, dst = [], []
    for _, row in df.iterrows():
        if row['rating'] >= rating_threshold:
            src.append(row['userId'])
            dst.append(row['movieId'] + num_users)      # item indices after user indices
    return torch.tensor( [src, dst], dtype=torch.long )

edge_index = create_edge_index( rating_df )
print(edge_index)

# Split indices into train/val/test set (label for train/val/test set)
train_indices, test_indices = train_test_split(range(edge_index.size(1)), test_size=0.2)
val_indices, test_indices = train_test_split(test_indices, test_size=0.5)

train_edge_index = edge_index[:, train_indices]     # indices 들이 모두 저장되어 있으므로 앞에는 무조건 : 
val_edge_index = edge_index[:, val_indices]
test_edge_index = edge_index[:, test_indices]

In [ ]:
### 3. NGCF Layer class

class NGCFLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()

        self.W1 = nn.Linear( input_dim, output_dim )
        self.W2 = nn.Linear( input_dim, output_dim )
        # self.dropout = nn.Dropout(dropout)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, edge_index, node_features, user_num, item_num):
        '''
        edge_index : 엣지 정보 (src, dst)의 집합.
        node_features: node별 이전 layer에서 생성된 벡터 정보가 담긴 matrix (H^(l-1)) (|V| * d)
        '''

        src, dst = edge_index       # src : user, dst : movie

        # calculate node degree (해당 node(user + movie)에 연결된 edge 수)
        deg = torch.zeros( node_features.size(0), device=node_features.device )     # inti with 0
        deg.index_add_( 0, src, torch.ones_like(src, dtype=torch.float) )           # calculate user degree
        deg.index_add_( 0, dst, torch.ones_like(dst, dtype=torch.float) )           # calculate movie degree

        # calculate 1/(root(deg(u)) * root(deg(i))) for all edge
        norm = 1.0 / torch.sqrt( deg[src] * deg[dst] )                              # 너무 인기 영화의 값이 커서 정규화

        src_feat = node_features[ src ] # H_u                                       # 각 edge 의 user embedding
        dst_feat = node_features[ dst ] # H_i                                       # 각 edge 의 item embedding

        edge_messages_for_src = self.W1( dst_feat ) + self.W2( dst_feat * src_feat )    # user 가 item (movie) 로 부터 받은 message 계산
        edge_messages_for_src *= norm.unsqueeze(1)

        edge_messages_for_dst = self.W1( src_feat ) + self.W2( src_feat * dst_feat )    # item 이 user 로 부터 받은 message 계산
        edge_messages_for_dst *= norm.unsqueeze(1)

        aggregated_messages = torch.zeros_like( node_features )                         # message 를 user 별로 합산
        aggregated_messages.index_add( 0, src, edge_messages_for_src )                  # user 마다 연결된 item 의 message 를 누적
        aggregated_messages[ :user_num ] += self.W1( node_features[ :user_num ] )       # user 자신의 embedding 도 더함

        aggregated_messages.index_add_( 0, dst, edge_messages_for_dst )                 # movie 마다 연결된 user 의 message 를 누적
        aggregated_messages[ user_num: ] += self.W1( node_features[ user_num: ] )       # movie 자신의 embedding 도 더함

        aggregated_features = self.leaky_relu(aggregated_messages)                      # ReLU

        # engineering approach
        # aggregated_features = self.dropout(aggregated_features)
        return aggregated_features

In [ ]:
### 4. NGCF 모델 class

class NGCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, layer_dims, dropout=0.1):       # embedding_dim = 64, layer_dims = 2
        super().__init__()

        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.node_embeddings = nn.Embedding( self.num_users + self.num_items, self.embedding_dim )
        nn.init.xavier_uniform_( self.node_embeddings.weight )

        self.layers = nn.ModuleList([
            NGCFLayer( input_dim=(embedding_dim if i == 0 else layer_dims[i - 1] ),
                       output_dim=layer_dims[i], dropout=dropout )
            for i in range(len(layer_dims))
        ])

    def forward(self, edge_index):
        node_features = self.node_embeddings.weight
        layer_outputs = [node_features]

        for layer in self.layers:
            node_features = layer( edge_index, node_features, self.num_users, self.num_items )
            layer_outputs.append( node_features )

        # Hint: NGCF의 final feature(representation)은 layer 별 feature에 대한 concatenated vector
        # Hint: 최종 final feature matrix에는 [feuture_vector for users + feature_vector for items]가 들어있음.

        final_features = torch.concat( layer_outputs, dim=-1 )
        
        user_features = final_features[ :self.num_users ]
        item_features = final_features[ self.num_users: 
        ]
        return user_features, item_features

    def bpr_loss( self, user_emb, pos_item_emb, neg_item_emb, reg_weight=1e-4 ):
        pos_scores = torch.sum( user_emb * pos_item_emb, dim=1 )
        neg_scores = torch.sum( user_emb * neg_item_emb, dim=1 )

        loss = -torch.mean( F.logsigmoid( pos_scores - neg_scores ) )
        reg_loss = reg_weight * ( user_emb.norm(2).pow(2) + pos_item_emb.norm(2).pow(2) + neg_item_emb.norm(2).pow(2) ) / user_emb.size(0)
        
        return loss + reg_loss

In [ ]:
### 5. Train

def train( model, optimizer, train_edge_index, val_edge_index, num_epochs, batch_size, device, k ):
    model.to(device)
    train_edge_index = train_edge_index.to(device)
    val_edge_index = val_edge_index.to(device)

    for epoch in range(num_epochs):
        model.train()

        total_loss = 0
        num_batches = len(train_edge_index[0]) // batch_size

        for _ in range(num_batches):
            optimizer.zero_grad()

            indices = torch.randint( 0, train_edge_index.size(1), (batch_size,), device=device )

            user_indices = train_edge_index[ 0, indices ]
            pos_item_indices = train_edge_index[ 1, indices ] - num_users
            neg_item_indices = torch.randint( 0, num_movies, (batch_size,), device=device )

            user_features, item_features = model( train_edge_index )

            u_emb = user_features[ user_indices ]
            pos_emb = item_features[ pos_item_indices ]
            neg_emb = item_features[ neg_item_indices ]

            loss = model.bpr_loss( u_emb, pos_emb, neg_emb )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {total_loss / num_batches:.4f}")

        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                user_features, item_features = model( train_edge_index )        # validation 도 train_edge_index 사용
                recall, precision, ndcg = evaluate( user_features.cpu(), item_features.cpu(), val_edge_index, k )
                print(f"[Validation] Epoch {epoch + 1}: Recall@{k}: {recall:.4f}, Precision@{k}: {precision:.4f}, NDCG@{k}: {ndcg:.4f}")

In [ ]:
### 6. Evaluation
    - Evalutation, Test 에서도 동일한 train_edge_index 사용

def evaluate( user_features, item_features, test_edge_index, k ):
    user_pos_items = defaultdict(list)
    
    E_test = test_edge_index.size(1)
    for i in range(E_test):
        u = test_edge_index[ 0, i ].item()                  # 첫번째 row : user index
        it = test_edge_index[ 1, i ].item() - num_users     # 두번째 row : movie index
        user_pos_items[ u ].append( it )                    # user 별로 본 영화의 index 목록이 dict 로 저장됨

    recalls, precisions, ndcgs = [], [], []

    for user, pos_items in user_pos_items.items():
        user_emb = user_features[ user ]
        scores = torch.matmul( item_features, user_emb )
        topk_scores, topk_indices = torch.topk( scores, k=k )
        topk_indices = topk_indices.cpu().numpy().tolist()

        hits = 0
        dcg = 0.0
        idcg = 0.0
        n_pos = len(pos_items)

        for rank, item_idx in enumerate( topk_indices ):
            if item_idx in pos_items:
                hits += 1
                dcg += 1.0 / math.log2(rank + 2)

        for rank in range( min(n_pos, k) ):
            idcg += 1.0 / math.log2(rank + 2)

        recall_u = hits / n_pos
        precision_u = hits / k
        ndcg_u = dcg / idcg if idcg > 0 else 0.0

        recalls.append(recall_u)
        precisions.append(precision_u)
        ndcgs.append(ndcg_u)

    recall = np.mean(recalls)
    precision = np.mean(precisions)
    ndcg = np.mean(ndcgs)

    return recall, precision, ndcg

In [ ]:
### 7. Test
    - Test 에서도 동일하게 train_edge_index 사용해서 user, item feature map 을 가져옴

def test( model, train_edge_index, test_edge_index, k, device ):
    model.eval()
    train_edge_index = train_edge_index.to(device)
    test_edge_index = test_edge_index.to(device)

    with torch.no_grad():
        user_features, item_features = model( train_edge_index )

    user_features = user_features.cpu()
    item_features = item_features.cpu()

    recall, precision, ndcg = evaluate( user_features, item_features, test_edge_index, k )
    recall = round(recall, 4)
    precision = round(precision, 4)
    ndcg = round(ndcg, 4)

    print(f"Recall@{k}: {recall:.4f}, Precision@{k}: {precision:.4f}, NDCG@{k}: {ndcg:.4f}")
    return recall, precision, ndcg


ngcf_model = NGCF( num_users, num_movies, embedding_dim=64, layer_dims=[64, 64], dropout=0.1 )
optimizer_ngcf = torch.optim.Adam( ngcf_model.parameters(), lr=1e-3 )

print("===== Train NGCF =====")
train(
    model=ngcf_model,
    optimizer=optimizer_ngcf,
    train_edge_index=train_edge_index,
    val_edge_index=val_edge_index,
    num_epochs=30,
    batch_size=1024,
    device=device,
    k=10
)

print("===== Test NGCF =====")
test(
    model=ngcf_model,
    train_edge_index=train_edge_index,
    test_edge_index=test_edge_index,
    k=10,
    device=device
)